# Final Experiment Notebook: Cache-Aware Channel Selection for MobileNetV2 on CIFAR-100

This notebook is a **robust comparison of conventional vs. cache-aware channel-selection policies** for MobileNetV2. It addresses the earlier issues by:
- discovering layers dynamically instead of assuming fixed internal indexing;
- creating a validated candidate table with explicit column names;
- limiting pruning per layer so no layer is completely masked;
- making cache pressure affect **layer-level pruning allocation**;
- checking model outputs before training and reporting the overlap between policies.

**Research honesty:** to keep this notebook compatible across torchvision versions, it applies channel-output masks at runtime rather than physically rebuilding a smaller network. This allows a controlled test of selection policies and accuracy, but it does **not** guarantee lower latency, parameter count, or energy. The cache pressure is an analytical activation-working-set proxy, not measured cache misses. Do not present it as hardware-proven energy savings.

## 1. Configuration

In [1]:
import os, copy, time, random, platform, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 0
IMAGE_SIZE = 224
NUM_CLASSES = 100
PRUNE_RATIO = 0.10          # total eligible channels selected for masking
MAX_LAYER_PRUNE_RATIO = 0.20  # safety cap: at most 20% in any eligible layer
CACHE_BUDGET_MB = 0.25      # analytical proxy threshold; report and sensitivity-test it
CACHE_WEIGHT_FLOOR = 0.25   # ensures every layer can receive some allocation
BASELINE_EPOCHS = 30      # increase substantially for final research results
FINETUNE_EPOCHS = 20
LR = 1e-4
USE_PRETRAINED = True
DATA_ROOT = "./data"
RESULTS_DIR = "./results_final"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

/home/komal-sai-jakka/PythonEnvs/ml_env/lib/python3.14/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Python: 3.14.4
PyTorch: 2.12.1+cu130
Device: cuda


## 2. Load CIFAR-100

In [2]:
# ImageNet normalization is appropriate when starting from ImageNet-pretrained weights.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMAGE_SIZE, padding=8),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_ds = datasets.CIFAR100(DATA_ROOT, train=True, download=True, transform=train_transform)
test_ds = datasets.CIFAR100(DATA_ROOT, train=False, download=True, transform=test_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE.type == "cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=(DEVICE.type == "cuda"))
print(f"Train: {len(train_ds):,}; test: {len(test_ds):,}")

Train: 50,000; test: 10,000


## 3. Build MobileNetV2

In [3]:
def make_model(num_classes=NUM_CLASSES, pretrained=USE_PRETRAINED):
    model = None
    if pretrained:
        try:
            model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
            print("Loaded ImageNet-pretrained MobileNetV2.")
        except Exception as exc:
            warnings.warn(f"Could not load pretrained weights ({type(exc).__name__}: {exc}); using random initialization.")
    if model is None:
        try:
            model = models.mobilenet_v2(weights=None)
        except TypeError:
            model = models.mobilenet_v2(pretrained=False)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model.to(DEVICE)

baseline = make_model()
print("Classifier:", baseline.classifier[-1])

Loaded ImageNet-pretrained MobileNetV2.
Classifier: Linear(in_features=1280, out_features=100, bias=True)


## 4. Training and evaluation

In [4]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
        n = labels.size(0)
        total_loss += loss.item() * n
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += n
    return total_loss / max(total, 1), correct / max(total, 1)

def fit(model, loader, epochs, lr=LR, label="model"):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    history = []
    for epoch in range(epochs):
        loss, acc = run_epoch(model, loader, optimizer)
        row = {"model": label, "epoch": epoch + 1, "train_loss": loss, "train_accuracy": acc}
        history.append(row)
        print(f"{label} | epoch {epoch+1}/{epochs} | loss={loss:.4f} | accuracy={acc:.4f}")
    return pd.DataFrame(history)

@torch.no_grad()
def evaluate(model, loader):
    loss, accuracy = run_epoch(model, loader)
    return {"test_loss": loss, "test_accuracy": accuracy}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

baseline_history = fit(baseline, train_loader, BASELINE_EPOCHS, label="baseline")
baseline_metrics = evaluate(baseline, test_loader)
print("Baseline:", baseline_metrics)

baseline | epoch 1/30 | loss=2.1439 | accuracy=0.4808
baseline | epoch 2/30 | loss=1.0211 | accuracy=0.7067
baseline | epoch 3/30 | loss=0.7724 | accuracy=0.7708
baseline | epoch 4/30 | loss=0.6268 | accuracy=0.8091
baseline | epoch 5/30 | loss=0.5208 | accuracy=0.8396
baseline | epoch 6/30 | loss=0.4432 | accuracy=0.8629
baseline | epoch 7/30 | loss=0.3746 | accuracy=0.8844
baseline | epoch 8/30 | loss=0.3212 | accuracy=0.9003
baseline | epoch 9/30 | loss=0.2751 | accuracy=0.9141
baseline | epoch 10/30 | loss=0.2321 | accuracy=0.9264
baseline | epoch 11/30 | loss=0.2068 | accuracy=0.9350
baseline | epoch 12/30 | loss=0.1841 | accuracy=0.9425
baseline | epoch 13/30 | loss=0.1631 | accuracy=0.9496
baseline | epoch 14/30 | loss=0.1417 | accuracy=0.9562
baseline | epoch 15/30 | loss=0.1282 | accuracy=0.9599
baseline | epoch 16/30 | loss=0.1143 | accuracy=0.9647
baseline | epoch 17/30 | loss=0.1057 | accuracy=0.9671
baseline | epoch 18/30 | loss=0.0979 | accuracy=0.9697
baseline | epoch 19

## 5. Discover eligible expansion convolutions robustly

For each MobileNetV2 `InvertedResidual` block, the notebook searches for ordinary (`groups == 1`) convolutions that expand channel width (`out_channels > in_channels`). It then identifies the immediate child module inside the block's convolution sequence that contains that convolution. The child module is where the channel-output mask will be applied, avoiding assumptions such as `block.conv[0][0]`.

In [5]:
def discover_expansion_convs(model):
    found = []
    for block_idx, block in enumerate(model.features):
        if block.__class__.__name__ != "InvertedResidual":
            continue
        conv_sequence = getattr(block, "conv", None)
        if conv_sequence is None:
            continue
        for conv_name, conv in block.named_modules():
            if not isinstance(conv, nn.Conv2d):
                continue
            if conv.groups != 1 or conv.out_channels <= conv.in_channels:
                continue

            # Find the immediate child of block.conv that contains this convolution.
            target_component = None
            target_local_name = None
            for child_name, child in conv_sequence.named_children():
                if any(sub is conv for sub in child.modules()):
                    target_component = child
                    target_local_name = child_name
                    break
            if target_component is None:
                # If the convolution is itself a direct child of the sequence, it is a valid target.
                for child_name, child in conv_sequence.named_children():
                    if child is conv:
                        target_component, target_local_name = child, child_name
                        break
            if target_component is None:
                continue

            found.append({
                "block_index": int(block_idx),
                "module_name": f"features.{block_idx}.conv.{target_local_name}",
                "conv": conv,
                "target_module": target_component,
                "in_channels": int(conv.in_channels),
                "out_channels": int(conv.out_channels),
            })

    unique, seen = [], set()
    for item in found:
        key = (id(item["conv"]), id(item["target_module"]))
        if key not in seen:
            unique.append(item)
            seen.add(key)
    return unique

eligible = discover_expansion_convs(baseline)
print("Eligible expansion convolutions:", len(eligible))
if not eligible:
    print("torchvision version:", __import__("torchvision").__version__)
    print("features[1]:")
    print(baseline.features[1])
    raise RuntimeError("No eligible expansion convolutions were discovered. Share the printed structure to diagnose the installed torchvision layout.")
display(pd.DataFrame([{k:v for k,v in item.items() if k not in ("conv", "target_module")} for item in eligible]))

Eligible expansion convolutions: 16


,block_index,module_name,in_channels,out_channels
0,2,features.2.conv.0,16,96
1,3,features.3.conv.0,24,144
2,4,features.4.conv.0,24,144
3,5,features.5.conv.0,32,192
4,6,features.6.conv.0,32,192
5,7,features.7.conv.0,32,192
6,8,features.8.conv.0,64,384
7,9,features.9.conv.0,64,384
8,10,features.10.conv.0,64,384
9,11,features.11.conv.0,64,384


## 6. Estimate per-layer activation working set and build candidates

In [6]:
def collect_target_output_shapes(model, eligible, image_size=IMAGE_SIZE):
    shapes = {}
    handles = []
    for item in eligible:
        key = item["module_name"]
        def make_hook(k):
            def hook(module, inputs, output):
                if isinstance(output, torch.Tensor) and output.ndim == 4:
                    shapes[k] = tuple(int(v) for v in output.shape)
            return hook
        handles.append(item["target_module"].register_forward_hook(make_hook(key)))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, 3, image_size, image_size, device=DEVICE))
    for handle in handles:
        handle.remove()
    model.train(was_training)
    return shapes

target_shapes = collect_target_output_shapes(baseline, eligible)
rows = []
for item in eligible:
    conv = item["conv"]
    shape = target_shapes.get(item["module_name"])
    if shape is None:
        raise RuntimeError(f"No activation shape captured for {item['module_name']}; cannot estimate working set.")
    _, channels, height, width = shape
    if channels != conv.out_channels:
        raise RuntimeError(f"Channel mismatch at {item['module_name']}: output has {channels}, conv has {conv.out_channels}.")
    ws_bytes = int(channels * height * width * 4)  # FP32 proxy, per sample
    ws_mb = ws_bytes / (1024 ** 2)
    pressure = max(ws_mb - CACHE_BUDGET_MB, 0.0) / max(ws_mb, 1e-12)
    filter_l1 = conv.weight.detach().abs().flatten(1).sum(1).cpu().numpy()
    for channel_idx, importance in enumerate(filter_l1):
        rows.append({
            "block_index": item["block_index"],
            "module_name": item["module_name"],
            "channel_index": int(channel_idx),
            "importance": float(importance),
            "out_channels": int(conv.out_channels),
            "height": int(height),
            "width": int(width),
            "working_set_bytes": ws_bytes,
            "working_set_mb": ws_mb,
            "cache_pressure": float(pressure),
        })

CANDIDATE_COLUMNS = ["block_index", "module_name", "channel_index", "importance",
                     "out_channels", "height", "width", "working_set_bytes",
                     "working_set_mb", "cache_pressure"]
candidates = pd.DataFrame(rows, columns=CANDIDATE_COLUMNS)
if candidates.empty or candidates["importance"].isna().any():
    raise RuntimeError("Candidate table is empty or importance contains missing values.")
print("Candidates:", len(candidates))
print("Columns:", candidates.columns.tolist())
print("Distinct cache-pressure values:", candidates["cache_pressure"].nunique())
display(candidates.groupby("module_name").agg(
    channels=("channel_index", "count"),
    working_set_mb=("working_set_mb", "first"),
    cache_pressure=("cache_pressure", "first"),
    min_importance=("importance", "min"),
    max_importance=("importance", "max")
).reset_index())

Candidates: 7104
Columns: ['block_index', 'module_name', 'channel_index', 'importance', 'out_channels', 'height', 'width', 'working_set_bytes', 'working_set_mb', 'cache_pressure']
Distinct cache-pressure values: 6


,module_name,channels,working_set_mb,cache_pressure,min_importance,max_importance
0,features.10.conv.0,384,0.287109,0.129252,2.688619e+00,8.717444
1,features.11.conv.0,384,0.287109,0.129252,3.817516e+00,8.446062
2,features.12.conv.0,576,0.430664,0.419501,3.938309e+00,8.280432
3,features.13.conv.0,576,0.430664,0.419501,1.215627e-13,8.152987
4,features.14.conv.0,576,0.430664,0.419501,3.991463e+00,10.735321
5,features.15.conv.0,960,0.179443,0.000000,1.901252e-12,11.403287
6,features.16.conv.0,960,0.179443,0.000000,4.644977e+00,12.762551
7,features.17.conv.0,960,0.179443,0.000000,7.290649e+00,12.742748
8,features.2.conv.0,96,4.593750,0.945578,1.517714e+00,4.956387
9,features.3.conv.0,144,1.722656,0.854875,8.947426e-16,5.491921


## 7. Select channels with a per-layer safety cap

In [7]:
def allocate_layer_quotas(layer_table, target_total, policy, max_layer_ratio=MAX_LAYER_PRUNE_RATIO):
    # layer_table has one row per eligible layer, with n_channels and cache_pressure.
    quota = {row.module_name: 0 for row in layer_table.itertuples()}
    caps = {
        row.module_name: min(int(math.floor(row.n_channels * max_layer_ratio)),
                             max(0, row.n_channels - 1))
        for row in layer_table.itertuples()
    }
    capacity = sum(caps.values())
    target_total = min(int(target_total), capacity)
    if target_total <= 0:
        return quota

    meta = {row.module_name: row for row in layer_table.itertuples()}
    # Greedy weighted fair allocation: conventional uses channel count as weight;
    # cache-aware upweights layers whose estimated activation working set exceeds budget.
    for _ in range(target_total):
        eligible_names = [name for name in quota if quota[name] < caps[name]]
        if not eligible_names:
            break
        def priority(name):
            row = meta[name]
            base = float(row.n_channels)
            if policy == "cache_aware":
                base *= (CACHE_WEIGHT_FLOOR + float(row.cache_pressure))
            return base / (quota[name] + 1.0)
        chosen = max(eligible_names, key=priority)
        quota[chosen] += 1
    return quota

def select_channels(df, policy, ratio=PRUNE_RATIO):
    required = set(CANDIDATE_COLUMNS)
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing candidate columns: {sorted(missing)}")
    if policy not in {"conventional", "cache_aware"}:
        raise ValueError("policy must be conventional or cache_aware")
    layer_table = df.groupby("module_name", as_index=False).agg(
        n_channels=("channel_index", "count"),
        cache_pressure=("cache_pressure", "first")
    )
    target = max(1, int(round(len(df) * ratio)))
    quotas = allocate_layer_quotas(layer_table, target, policy)
    selected_parts = []
    for layer_name, quota in quotas.items():
        if quota <= 0:
            continue
        group = df[df["module_name"] == layer_name].sort_values(
            ["importance", "channel_index"], ascending=[True, True]
        )
        selected_parts.append(group.head(quota))
    if not selected_parts:
        raise RuntimeError("No channels selected. Reduce PRUNE_RATIO or increase MAX_LAYER_PRUNE_RATIO.")
    selected = pd.concat(selected_parts, ignore_index=True)
    return selected

import math
conventional_selected = select_channels(candidates, "conventional")
cache_selected = select_channels(candidates, "cache_aware")
print("Requested target:", round(len(candidates) * PRUNE_RATIO))
print("Conventional selected:", len(conventional_selected))
print("Cache-aware selected:", len(cache_selected))
overlap = len(set(zip(conventional_selected.module_name, conventional_selected.channel_index)) &
              set(zip(cache_selected.module_name, cache_selected.channel_index)))
print(f"Selection overlap: {overlap} / {len(conventional_selected)} ({overlap/max(len(conventional_selected),1):.1%})")
display(pd.DataFrame([
    {"policy":"conventional", "selected":len(conventional_selected),
     "layers_touched":conventional_selected.module_name.nunique(),
     "mean_importance":conventional_selected.importance.mean()},
    {"policy":"cache_aware", "selected":len(cache_selected),
     "layers_touched":cache_selected.module_name.nunique(),
     "mean_importance":cache_selected.importance.mean()},
]))

Requested target: 710
Conventional selected: 710
Cache-aware selected: 710
Selection overlap: 549 / 710 (77.3%)


,policy,selected,layers_touched,mean_importance
0,conventional,710,16,5.172161
1,cache_aware,710,16,4.584834


## 8. Apply masks to intermediate activations (shape-safe)

In [8]:
def install_activation_masks(model, selected_df):
    modules = dict(model.named_modules())
    masks = {}
    for module_name, group in selected_df.groupby("module_name"):
        module = modules.get(module_name)
        if module is None:
            raise RuntimeError(f"Could not resolve target module {module_name} in copied model.")
        channel_count = int(group["out_channels"].iloc[0])
        mask = torch.ones(channel_count, device=DEVICE)
        idx = group["channel_index"].astype(int).to_list()
        if any(i < 0 or i >= channel_count for i in idx):
            raise ValueError(f"Invalid channel index in {module_name}")
        mask[idx] = 0.0
        masks[module_name] = mask
    handles = []
    for module_name, mask in masks.items():
        module = modules[module_name]
        def make_hook(channel_mask):
            def hook(_module, _inputs, output):
                if not isinstance(output, torch.Tensor) or output.ndim != 4:
                    raise RuntimeError("Expected a 4-D convolutional activation for channel masking.")
                if output.shape[1] != channel_mask.numel():
                    raise RuntimeError(f"Mask/output channel mismatch: {output.shape[1]} vs {channel_mask.numel()}")
                return output * channel_mask.to(device=output.device, dtype=output.dtype).view(1, -1, 1, 1)
            return hook
        handles.append(module.register_forward_hook(make_hook(mask)))
    return handles

def make_masked_model(base_model, selected_df):
    model = copy.deepcopy(base_model).to(DEVICE)
    handles = install_activation_masks(model, selected_df)
    # Verify forward shape before training.
    model.eval()
    with torch.no_grad():
        output = model(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))
    if tuple(output.shape) != (2, NUM_CLASSES):
        for h in handles: h.remove()
        raise RuntimeError(f"Unexpected output shape: {tuple(output.shape)}")
    return model, handles

conventional_model, conventional_handles = make_masked_model(baseline, conventional_selected)
cache_model, cache_handles = make_masked_model(baseline, cache_selected)
print("Both masked models passed the forward-pass shape check.")

Both masked models passed the forward-pass shape check.


## 9. Fine-tune under identical conditions

In [9]:
conventional_history = fit(conventional_model, train_loader, FINETUNE_EPOCHS, label="conventional_masked")
cache_history = fit(cache_model, train_loader, FINETUNE_EPOCHS, label="cache_aware_masked")

conventional_masked | epoch 1/20 | loss=0.1446 | accuracy=0.9528
conventional_masked | epoch 2/20 | loss=0.0896 | accuracy=0.9715
conventional_masked | epoch 3/20 | loss=0.0755 | accuracy=0.9758
conventional_masked | epoch 4/20 | loss=0.0716 | accuracy=0.9777
conventional_masked | epoch 5/20 | loss=0.0726 | accuracy=0.9762
conventional_masked | epoch 6/20 | loss=0.0624 | accuracy=0.9802
conventional_masked | epoch 7/20 | loss=0.0611 | accuracy=0.9801
conventional_masked | epoch 8/20 | loss=0.0606 | accuracy=0.9805
conventional_masked | epoch 9/20 | loss=0.0551 | accuracy=0.9824
conventional_masked | epoch 10/20 | loss=0.0570 | accuracy=0.9816
conventional_masked | epoch 11/20 | loss=0.0534 | accuracy=0.9834
conventional_masked | epoch 12/20 | loss=0.0508 | accuracy=0.9839
conventional_masked | epoch 13/20 | loss=0.0508 | accuracy=0.9844
conventional_masked | epoch 14/20 | loss=0.0490 | accuracy=0.9842
conventional_masked | epoch 15/20 | loss=0.0449 | accuracy=0.9857
conventional_masked

## 10. Accuracy, model size, and selection diagnostics

In [10]:
def zero_fraction_in_masked_activations(selected_df):
    # Reports selected channels / all eligible channels; not a parameter sparsity metric.
    return len(selected_df) / max(len(candidates), 1)

comparison_rows = []
for label, model, selected in [
    ("baseline", baseline, candidates.iloc[0:0]),
    ("conventional_masked", conventional_model, conventional_selected),
    ("cache_aware_masked", cache_model, cache_selected),
]:
    metrics = evaluate(model, test_loader)
    comparison_rows.append({
        "model": label,
        "test_accuracy": metrics["test_accuracy"],
        "test_loss": metrics["test_loss"],
        "parameter_count": count_parameters(model),
        "parameter_count_millions": count_parameters(model) / 1e6,
        "selected_channel_fraction": zero_fraction_in_masked_activations(selected),
    })
results_df = pd.DataFrame(comparison_rows)
display(results_df)

,model,test_accuracy,test_loss,parameter_count,parameter_count_millions,selected_channel_fraction
0,baseline,0.8015,0.945938,2351972,2.351972,0.000000
1,conventional_masked,0.7978,1.066201,2351972,2.351972,0.099944
2,cache_aware_masked,0.7966,1.063522,2351972,2.351972,0.099944


## 11. Inference timing (interpret cautiously)

In [11]:
@torch.no_grad()
def benchmark_inference(model, loader, max_batches=40, warmup_batches=5):
    model.eval()
    samples, per_image_times = 0, []
    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= max_batches:
            break
        images = images.to(DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model(images)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        samples += images.size(0)
        if batch_idx >= warmup_batches:
            per_image_times.append(elapsed / images.size(0))
    return {
        "mean_seconds_per_image": float(np.mean(per_image_times)) if per_image_times else float("nan"),
        "median_seconds_per_image": float(np.median(per_image_times)) if per_image_times else float("nan"),
        "timed_batches": max(0, min(max_batches, len(loader)) - warmup_batches),
        "device": str(DEVICE),
    }

timing_rows = []
for label, model in [
    ("baseline", baseline),
    ("conventional_masked", conventional_model),
    ("cache_aware_masked", cache_model),
]:
    timing_rows.append({"model": label, **benchmark_inference(model, test_loader)})
timing_df = pd.DataFrame(timing_rows)
display(timing_df)
print("Runtime hooks can add overhead; these timings are not evidence of speedup or energy reduction.")

,model,mean_seconds_per_image,median_seconds_per_image,timed_batches,device
0,baseline,0.000828,0.000818,35,cuda
1,conventional_masked,0.000952,0.000952,35,cuda
2,cache_aware_masked,0.000952,0.000952,35,cuda


Runtime hooks can add overhead; these timings are not evidence of speedup or energy reduction.


## 12. Export all result tables

In [12]:
os.makedirs(RESULTS_DIR, exist_ok=True)
candidates.to_csv(os.path.join(RESULTS_DIR, "candidates.csv"), index=False)
conventional_selected.to_csv(os.path.join(RESULTS_DIR, "conventional_selected.csv"), index=False)
cache_selected.to_csv(os.path.join(RESULTS_DIR, "cache_aware_selected.csv"), index=False)
results_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison.csv"), index=False)
timing_df.to_csv(os.path.join(RESULTS_DIR, "inference_timing.csv"), index=False)
conventional_history.to_csv(os.path.join(RESULTS_DIR, "conventional_training_history.csv"), index=False)
cache_history.to_csv(os.path.join(RESULTS_DIR, "cache_aware_training_history.csv"), index=False)
baseline_history.to_csv(os.path.join(RESULTS_DIR, "baseline_training_history.csv"), index=False)
print("Saved CSV results to:", os.path.abspath(RESULTS_DIR))

Saved CSV results to: /home/komal-sai-jakka/ACADEMICS/5th_sem/machine_learning/research/mobilenet/results_final


## 13. Research interpretation and next steps

1. **Accuracy:** compare the pruned policies to the same trained baseline. A large collapse suggests the pruning ratio or fine-tuning budget is too aggressive.
2. **Fairness:** both policies use the same total channel budget, baseline checkpoint, training data, and fine-tuning epochs.
3. **Policy effect:** inspect selection overlap and per-layer quotas. If cache pressure is constant across layers, the cache-aware allocator cannot meaningfully differ; change/report `CACHE_BUDGET_MB` only with a justified sensitivity analysis.
4. **No physical compaction:** activation masks preserve the model's parameter count and tensor shapes. This notebook cannot establish lower FLOPs, latency, or energy.
5. **For a paper:** repeat with several seeds and pruning ratios, report mean ± standard deviation, validate cache estimates against hardware counters where possible, and use a dependency-aware physical compaction/deployment path before claiming speed or energy gains.